# HT/NALM vs PT/NALM — T cell donor comparison

Both cell systems use **NALM-6** (B-ALL line) co-cultured with T cells; the T-cell donor differs:

* **HT/NALM** = `NALM-6 + healthy T`
* **PT/NALM** = `NALM-6 + patient T`

Because the B compartment is held fixed, the analyses below focus on the **T cell side** — CD8 in particular.

Sample availability:

| Time × Condition | HT/NALM | PT/NALM |
| --- | --- | --- |
| 6h Mock          | S005 | S013 |
| 6h Blinatumomab  | S006 | S014 |
| 48h Mock         | S007 | — (missing) |
| 48h Blinatumomab | S008 | S016 |

PT/NALM has no 48h Mock, so cross-system analyses that need both Mock & Blina are run only at **6h**.

In [ ]:
# [0 · Imports & configuration]
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

sc.set_figure_params(dpi=100, frameon=False)
plt.rcParams['figure.max_open_warning'] = 0

print(f'scvi-tools: {scvi.__version__}')

from nalm_utils import *

CACHE_DIR        = Path('/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data/cache')
ANNOTATED_CACHE  = CACHE_DIR / 'adata_cytovi_annotated_compat.h5ad'

# Two systems being compared
SYS_HT = 'NALM-6 + healthy T'    # HT/NALM
SYS_PT = 'NALM-6 + patient T'    # PT/NALM

In [ ]:
# [1 · Data loading]
adata = sc.read_h5ad(ANNOTATED_CACHE)

# Sample availability across the two systems being compared
mask_nalm = adata.obs['cell_system'].isin([SYS_HT, SYS_PT])
print(f'Total cells in HT/NALM + PT/NALM: {mask_nalm.sum()}')
pd.crosstab(
    index=[adata.obs[mask_nalm]['cell_system'], adata.obs[mask_nalm]['sample']],
    columns=[adata.obs[mask_nalm]['time'], adata.obs[mask_nalm]['condition']],
)

In [ ]:
mask_6h_mock_t = (
    
    (adata.obs['cell_type_annot'].isin(['CD4', 'CD8','B'])) &
    (adata.obs['cell_system'].isin([SYS_HT, SYS_PT]))
)
adata_6h = adata[mask_6h_mock_t].copy()

print(f'6h Mock T cells (healthy-T systems): {adata_6h.n_obs}')
pd.crosstab(index=adata_6h.obs['sample'], columns=[adata_6h.obs['cell_system'], adata_6h.obs['cell_type_annot']])

sc.pl.umap(adata_6h,                                                       
  color=['CD3e','CD19','CD8','cell_system','cell_type_annot','condition'],          
  layer='arcsinh', frameon=False)

## Cross-condition comparison — CD8 across time × condition

In [ ]:
# [2 · 4-way DA panel: CD8 — HT/NALM vs PT/NALM across time × condition]
# (PT/NALM has no 48h Mock, so that group is missing on the comparison side.)
# B cell markers are dropped to keep the focus on T-cell biology (NALM-6 contamination).
def _load_b_panel():
    for name in ('b_cell_markers', 'or'):
        try:
            return load_marker_panel(name)
        except KeyError:
            continue
    raise KeyError('No B cell marker panel found in marker_panels.json')

B_CELL_MARKERS_SET = set(_load_b_panel())

mask_ht_cd8 = (
    (adata.obs['cell_system'] == SYS_HT) &
    (adata.obs['cell_type_annot'] == 'CD8')
)
adata_ht_cd8 = adata[mask_ht_cd8].copy()
adata_ht_cd8.obs['time_cond'] = (
    adata_ht_cd8.obs['time'].astype(str) + ' ' + adata_ht_cd8.obs['condition'].astype(str)
)
adata_ht_cd8 = adata_ht_cd8[:, ~adata_ht_cd8.var_names.isin(B_CELL_MARKERS_SET)].copy()

mask_pt_cd8 = (
    (adata.obs['cell_system'] == SYS_PT) &
    (adata.obs['cell_type_annot'] == 'CD8')
)
adata_pt_cd8 = adata[mask_pt_cd8].copy()
adata_pt_cd8.obs['time_cond'] = (
    adata_pt_cd8.obs['time'].astype(str) + ' ' + adata_pt_cd8.obs['condition'].astype(str)
)
adata_pt_cd8 = adata_pt_cd8[:, ~adata_pt_cd8.var_names.isin(B_CELL_MARKERS_SET)].copy()

print(f'CD8 HT/NALM: {adata_ht_cd8.n_obs}')
print(adata_ht_cd8.obs['time_cond'].value_counts().to_string())
print(f'\nCD8 PT/NALM: {adata_pt_cd8.n_obs}')
print(adata_pt_cd8.obs['time_cond'].value_counts().to_string())

plot_marker_panel_violins(
    adata_ht_cd8, 'cd8_t_cell_markers',
    group_key='time_cond',
    adata_compare=adata_pt_cd8,
    primary_label='HT/NALM',
    compare_label='PT/NALM',
)

## LFC scatter — Blinatumomab vs Mock at 6h

PT/NALM has no 48h Mock, so the LFC (Blina/Mock) comparison is restricted to **6h**.

In [ ]:
# [7 · LFC scatter — Blina vs Mock, HT/NALM (x) vs PT/NALM (y), CD8, 6h]
fig, ax = plt.subplots(figsize=(8, 8))
plot_lfc_scatter(
    adata, time_val='6h',
    sys_a=SYS_HT, sys_b=SYS_PT,
    cell_type='CD8',
    label_a='HT/NALM', label_b='PT/NALM',
    color_above='#9467bd',  # higher LFC in PT/NALM (above y=x)
    color_below='#2ca02c',  # higher LFC in HT/NALM (below y=x)
    top_k=10,
    ax=ax,
)
plt.tight_layout()
plt.show()

## Mock vs Blinatumomab — abundance + spatial, per system

In [ ]:
# [8 · CD8 Blina vs Mock — abundance + spatial, per system, 6h]
# 12c-style 2x2 panel (abundance up/down, spatial coloc up/down) per system.
for sys_label, sys_val in [('HT/NALM', SYS_HT), ('PT/NALM', SYS_PT)]:
    plot_condition_comparison(
        adata,
        time_val='6h',
        cell_system=sys_val,
        cell_type='CD8',
        cond_a='Blinatumomab',
        cond_b='Mock',
        layer='arcsinh',
        obsm_key='spatial_asinh5',
        top_n=15,
        system_label=sys_label,
    )

## Spatial subsets for selected-marker comparisons

In [ ]:
# [9 · Spatial subsets — CD8 cells, per condition / system]
def _sp_subset(adata_full, time_val, cond_val, system_val):
    mask = (
        (adata_full.obs['time'] == time_val) &
        (adata_full.obs['condition'] == cond_val) &
        (adata_full.obs['cell_type_annot'] == 'CD8') &
        (adata_full.obs['cell_system'] == system_val)
    )
    sub = adata_full[mask]
    sp = sub.obsm['spatial_asinh5']
    if not isinstance(sp, pd.DataFrame):
        sp = pd.DataFrame(sp, index=sub.obs_names)
    print(f'  {time_val} {cond_val:15s} {system_val:25s} → {sp.shape[0]} cells')
    return sp

print('Building spatial subsets (CD8 only):')
sp_6h_mock_ht   = _sp_subset(adata, '6h',  'Mock',         SYS_HT)
sp_6h_mock_pt   = _sp_subset(adata, '6h',  'Mock',         SYS_PT)
sp_6h_blina_ht  = _sp_subset(adata, '6h',  'Blinatumomab', SYS_HT)
sp_6h_blina_pt  = _sp_subset(adata, '6h',  'Blinatumomab', SYS_PT)
sp_48h_blina_ht = _sp_subset(adata, '48h', 'Blinatumomab', SYS_HT)
sp_48h_blina_pt = _sp_subset(adata, '48h', 'Blinatumomab', SYS_PT)

all_sp_cols = sp_6h_mock_ht.columns

## CD8 immune synapse — HT/NALM vs PT/NALM at 6h Blinatumomab

In [ ]:
# [12 · CD8 immune synapse: heatmaps + networks at 6h Blina]
SYNAPSE_CATEGORIES = {
    'cSMAC (signaling core)': (['CD3e', 'CD8', 'CD2', 'CD28', 'CD134', 'CD137',
                                'CD226', 'TIGIT', 'CD279', 'VISTA'],       '#e41a1c'),
    'pSMAC (adhesion ring)':  (['CD11a', 'CD50', 'KLRG1', 'CD94', 'CD48',
                                'CD352', 'CD53'],                           '#4daf4a'),
    'Exclusion zone':         (['CD45', 'CD43', 'CD44'],                    '#377eb8'),
}

mat_syn_ht, mat_syn_pt, mat_syn_diff = plot_synapse_suite(
    sp_a=sp_6h_blina_ht, sp_b=sp_6h_blina_pt,
    categories=SYNAPSE_CATEGORIES,
    label_a='HT/NALM 6h Blina', label_b='PT/NALM 6h Blina',
    diff_label='Diff (HT − PT)',
    suite_name='Immune synapse',
    highlight_node='CD3e',
    var_filter=adata.var_names,
    cluster_k=[3, 5],
)

# B cells — HT/NALM vs PT/NALM

The B compartment (NALM-6) is identical across the two cell systems; only the
T-cell donor differs. The analyses below mirror the T-cell side and ask whether
the T donor (healthy vs patient) leaves a measurable imprint on NALM-6 state.

## Cross-condition comparison — B cells across time × condition

In [ ]:
# [B-2 · 4-way DA panel: B cells — HT/NALM vs PT/NALM across time × condition]
# (PT/NALM has no 48h Mock, so that group is missing on the comparison side.)
mask_ht_b = (
    (adata.obs['cell_system'] == SYS_HT) &
    (adata.obs['cell_type_annot'] == 'B')
)
adata_ht_b = adata[mask_ht_b].copy()
adata_ht_b.obs['time_cond'] = (
    adata_ht_b.obs['time'].astype(str) + ' ' + adata_ht_b.obs['condition'].astype(str)
)

mask_pt_b = (
    (adata.obs['cell_system'] == SYS_PT) &
    (adata.obs['cell_type_annot'] == 'B')
)
adata_pt_b = adata[mask_pt_b].copy()
adata_pt_b.obs['time_cond'] = (
    adata_pt_b.obs['time'].astype(str) + ' ' + adata_pt_b.obs['condition'].astype(str)
)

print(f'B HT/NALM: {adata_ht_b.n_obs}')
print(adata_ht_b.obs['time_cond'].value_counts().to_string())
print(f'\nB PT/NALM: {adata_pt_b.n_obs}')
print(adata_pt_b.obs['time_cond'].value_counts().to_string())

# Use the B-cell panel (currently keyed as 'or' in marker_panels.json).
plot_marker_panel_violins(
    adata_ht_b, 'or',
    group_key='time_cond',
    adata_compare=adata_pt_b,
    primary_label='HT/NALM',
    compare_label='PT/NALM',
)

## LFC scatter — Blinatumomab vs Mock at 6h, B cells

PT/NALM has no 48h Mock, so the LFC (Blina/Mock) comparison is restricted to **6h**.

In [ ]:
# [B-7 · LFC scatter — Blina vs Mock, HT/NALM (x) vs PT/NALM (y), B cells, 6h]
fig, ax = plt.subplots(figsize=(8, 8))
plot_lfc_scatter(
    adata, time_val='6h',
    sys_a=SYS_HT, sys_b=SYS_PT,
    cell_type='B',
    label_a='HT/NALM', label_b='PT/NALM',
    color_above='#9467bd',  # higher LFC in PT/NALM (above y=x)
    color_below='#2ca02c',  # higher LFC in HT/NALM (below y=x)
    top_k=10,
    ax=ax,
)
plt.tight_layout()
plt.show()

## Mock vs Blinatumomab — abundance + spatial, per system, B cells

In [ ]:
# [B-8 · B cell Blina vs Mock — abundance + spatial, per system, 6h]
# 12c-style 2x2 panel (abundance up/down, spatial coloc up/down) per system.
for sys_label, sys_val in [('HT/NALM', SYS_HT), ('PT/NALM', SYS_PT)]:
    plot_condition_comparison(
        adata,
        time_val='6h',
        cell_system=sys_val,
        cell_type='B',
        cond_a='Blinatumomab',
        cond_b='Mock',
        layer='arcsinh',
        obsm_key='spatial_asinh5',
        top_n=15,
        system_label=sys_label,
    )

## Spatial subsets — B cells, 6h Blina

In [ ]:
# [B-9 · Spatial subsets — B cells, 6h Blina HT/PT]
def _sp_subset_b(adata_full, time_val, cond_val, system_val):
    mask = (
        (adata_full.obs['time'] == time_val) &
        (adata_full.obs['condition'] == cond_val) &
        (adata_full.obs['cell_type_annot'] == 'B') &
        (adata_full.obs['cell_system'] == system_val)
    )
    sub = adata_full[mask]
    sp = sub.obsm['spatial_asinh5']
    if not isinstance(sp, pd.DataFrame):
        sp = pd.DataFrame(sp, index=sub.obs_names)
    print(f'  {time_val} {cond_val:15s} {system_val:25s} → {sp.shape[0]} cells')
    return sp

print('Building B-cell spatial subsets:')
sp_b_6h_blina_ht = _sp_subset_b(adata, '6h', 'Blinatumomab', SYS_HT)
sp_b_6h_blina_pt = _sp_subset_b(adata, '6h', 'Blinatumomab', SYS_PT)

all_sp_cols_b = sp_b_6h_blina_ht.columns

## B cell APC synapse — HT/NALM vs PT/NALM at 6h Blinatumomab

Antigen-presentation complex on the NALM-6 (B) side: MHC-II + costimulation +
inhibitory ligands + co-receptors + adhesion. Asks whether the T donor changes
how NALM-6 organises its synapse-facing surface.

In [ ]:
# [B-12 · B cell APC synapse: heatmaps + networks at 6h Blina]
APC_SYNAPSE_CATEGORIES = {
    'MHC Class II (Ag presentation)':  (['HLA-DR-DP-DQ', 'HLA-DR', 'HLA-DQ', 'HLA-ABC'], '#e41a1c'),
    'Costimulation':                    (['CD80', 'CD86', 'CD40'],                        '#ff7f00'),
    'Inhibitory / Checkpoint ligands':  (['CD274', 'CD273', 'CD32', 'CD72', 'CD305'],     '#377eb8'),
    'B cell co-receptors':              (['CD19', 'CD20', 'CD22', 'CD79a'],               '#4daf4a'),
    'Adhesion (pSMAC)':                 (['CD54', 'CD58', 'CD50', 'CD102'],               '#984ea3'),
}

mat_apc_ht, mat_apc_pt, mat_apc_diff = plot_synapse_suite(
    sp_a=sp_b_6h_blina_ht, sp_b=sp_b_6h_blina_pt,
    categories=APC_SYNAPSE_CATEGORIES,
    label_a='HT/NALM 6h Blina', label_b='PT/NALM 6h Blina',
    diff_label='Diff (HT − PT)',
    suite_name='APC synapse',
    highlight_node='HLA-DR-DP-DQ',
    var_filter=adata.var_names,
    cluster_k=[3, 5],
)